# Analysis for fragment linking

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from plotly import express as px
from tqdm import tqdm

In [ ]:
from tools import (
    compute_uniqueness,
    compute_novelty,
    compute_unique_novelty,
)

## Load data

In [ ]:
# load individually evaluation files

files = {
    "ground truth linker": "data/conditional_fragments/reference_mols_named.csv",
    "link replacement": "predictions/conditional_fragments/attempt_2/link_replacment.csv",
    "link replacement no h": "predictions/conditional_fragments/attempt_2/link_replacment_no_h.csv",
}

dfs = []
for method, file in files.items():
    print(f"Loading {method}")
    df = pd.read_csv(file)
    # Evaluation script does not break molecules correctly, introducting these rows
    df = df[~df.fail.fillna(0).astype(bool)].reset_index(drop=True)
    df = df.drop(columns=["index", "fail"], errors="ignore")

    df_frag = pd.read_csv(
        Path(file).parent / (Path(file).stem + "_combined.csv"),
        low_memory=False,
    ).reset_index()
    df_frag.columns = [c.lower().replace(" ", "_") for c in df_frag.columns]
    df_frag = df_frag.drop(columns=["index", "fail", "error"])

    if len(df) != len(df_frag):
        print(f"Lengths are not equal: {len(df)} != {len(df_frag)}")
        continue
    df = pd.concat([df, df_frag], axis=1, ignore_index=False)
    df["method"] = method

    if method == "ground truth linker":
        df = df[:1000]  # only used the first 1000 linkers

    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

df["total_number"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]

df["num_atoms_missing"] = df["num_atoms_link"] - df["num_atoms_frag"]
df["share_atoms_frag"] = df["num_atoms_frag"] / df["num_atoms_link"]
df["share_atom_missing"] = df["num_atoms_missing"] / df["num_atoms_link"]

In [ ]:
file = (
    "/homes/buttensc/Projects/semla-flow/data/conditional_fragments/reference_mols.csv"
)
truth = pd.read_csv(file)
truth = truth[truth.fail != 1.0]
print(len(truth))

In [ ]:
reference_smiles = set(truth["smiles"])
print(len(reference_smiles))

In [ ]:
# enrichment
df["total_number"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]
df["novel"] = df["smiles"].map(lambda x: x not in reference_smiles)
df["valid_novel"] = df["valid"] & df["novel"]
df["valid_smiles"] = df["valid"].astype(bool) * df["smiles"]
df["valid_scaffold_rdkit_csk"] = df["valid"].astype(bool) & df[
    "scaffold_rdkit_csk"
].astype(bool)
df["valid_scaffold_hop_smiles"] = (~df["valid_scaffold_rdkit_csk"]) * df["valid_smiles"]

In [ ]:
# define order of methods
order = [
    "ground truth linker",
    "link replacement",
    "link replacement no h",
]
df["method"] = pd.Categorical(df["method"], categories=order, ordered=True)

In [ ]:
# filter
aggs = {
    "sucos_frag": ("sucos_frag", "max"),
    "sucos_link": ("sucos_link", "max"),
    "scaffold_conserved": ("scaffold_rdkit_csk", "max"),
}
df_best = (
    df[(df.valid & df.novel) | ((df.method == "ground truth linker") & df.valid)]
    .groupby(["method", "reference_molecule", "smiles"], observed=True)
    .agg(**aggs)
    .reset_index()
)

## Tables

In [ ]:
df.columns

### Validity

In [ ]:
def mean(x):
    n = 100000
    if len(x) == 0:
        return float("nan")
    if len(x) > 2000:
        return np.sum(x) / n
    return np.sum(x) / 1127


def std(x):
    n = 100000
    if len(x) == 0:
        return float("nan")
    if len(x) > n:
        return np.std(x)
    m = np.sum(x) / n
    return np.sqrt(np.sum((x - m) ** 2) / n)


aggs = {
    "total_number": ("total_number", "sum"),
    "generated": ("total_number", mean),
    "connected": ("connected", mean),
    "chemical": ("chemical", mean),
    "physical": ("physical", mean),
    "valid": ("valid", mean),
    # "valid_scaffold_hop": ("scaffold_rdkit_csk", 1 - mean),
}
df_agg = df.groupby(["method"], observed=False).agg(**aggs)
cols = df_agg.columns
df_agg.style.format("{:.0f}", subset=cols[:1]).format("{:.1%}", subset=cols[1:])

### Properties

In [ ]:
df_filter = df[df.valid]
aggs = {
    "qed": ["mean", "std"],
    "sa": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
    "weight": ["mean", "std"],
    "num_heavy": ["mean", "std"],
    "num_rings": ["mean", "std"],
    "lipinski": ["mean", "std"],
    "logp": ["mean", "std"],
    "spacial": ["mean", "std"],
}
df_agg = df_filter.groupby(["method"], observed=False).agg(aggs)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

### Energy ratio

In [ ]:
df_filter = df[df.valid]
df_filter = df
aggs = {
    "ensemble_avg_energy": ["mean", "std"],
    "mol_pred_energy": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
}
df_agg = df_filter.groupby(["method"], observed=False).agg(aggs)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

### Unique, novel, useful

In [ ]:
n = 100000

df_filter = df
aggs = {
    "Valid": ("valid", lambda x: sum(x) / n if len(x) > 2000 else sum(x) / 1127),
    # "Valid & Scaffold Hop": ("scaffold_rdkit_csk", lambda x: 1 - mean(x)),
    # "Valid & Unique": (
    #     "valid_smiles",
    #     lambda x: compute_uniqueness(x, total=1) / n,
    # ),
    # "Valid & Novel": (
    #     "valid_smiles",
    #     lambda x: compute_novelty(x, reference_smiles, total=1) / n,
    # ),
    "Valid & Unique & Novel": (
        "valid_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1) / n
        if len(x) > 2000
        else compute_unique_novelty(x, reference_smiles, total=1) / 1127,
    ),
    "Valid & Unique & Novel & Scaffold Hop": (
        "valid_scaffold_hop_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1) / n
        if len(x) > 2000
        else compute_unique_novelty(x, reference_smiles, total=1) / 1127,
    ),
}
df_agg = df_filter.groupby(["method"], observed=False).agg(**aggs)
cols = df_agg.columns
df_agg.style.format("{:.2%}", subset=cols)

### Quality

In [ ]:
df_filter = df[df.valid]
df_filter = df
aggs = {
    "sucos_frag mean": ("sucos_frag", "mean"),
    "sucos_frag std": ("sucos_frag", "std"),
    "sucos_frag > 0.8 ": ("sucos_frag", lambda x: sum(x > 0.8) / n),
    "sucos_link mean": ("sucos_link", "mean"),
    "sucos_link std": ("sucos_link", "std"),
    "sucos_link > 0.55 ": ("sucos_link", lambda x: sum(x > 0.55) / n),
}
df_agg = df_filter.groupby(["method"], observed=False).agg(**aggs)
# df_agg["tanimoto > 0.8 and sucos > 0.55"] = (
#     df_agg["sucos > 0.55 "] * df_agg["tanimoto > 0.8 "]
# )
cols = df_agg.columns
df_agg.style.format("{:.2%}", subset=cols)

# Usefulness

In [ ]:
# How many unique new linkers capitulate the fragments and have SuCOS larger than x?

# How many good new linkers were created?
threshold_link = 0.7
threshold_frag = 0.7
aggs = {
    f"sucos_frag > {threshold_frag}": (
        "sucos_frag",
        lambda x: sum(x > threshold_frag) > 0,
    ),
    f"sucos_link > {threshold_link}": (
        "sucos_link",
        lambda x: sum(x > threshold_link) > 0,
    ),
}
df_agg = df_best.groupby(["method", "reference_molecule"], observed=True).agg(**aggs)
df_agg[f"sucos_link > {threshold_link} and sucos_frag > {threshold_frag}"] = (
    df_agg[f"sucos_link > {threshold_link}"] * df_agg[f"sucos_frag > {threshold_frag}"]
)
print("number of reference molecules new linkers were created for")
df_agg.groupby(["method"], observed=True).sum()
# df_agg

## Plots

In [ ]:
df_best

In [ ]:
fig = sns.histplot(
    df_best[df_best.method != "ground truth linker"]
    .groupby(["method", "smiles"], observed=True)
    .agg({"sucos_frag": "max"}),
    x="sucos_frag",
    hue="method",
    # complementary=True,
    common_norm=False,
    # stat="count",
    stat="density",
    element="step",
    # fill=False,
    # bins=50,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0, 1),
    # ylim=(0, 1400),
    title="How much do the Unique Novel Valid linkers resemble the fragments?",
    xlabel="Fragment SuCOS",
)
plt.savefig("plots/conditional_frags_comparison_to_fragments.png")

In [ ]:
fig = sns.histplot(
    df_best[df_best.method != "ground truth linker"]
    .groupby(["method", "smiles"], observed=True)
    .agg({"sucos_link": "max"}),
    x="sucos_link",
    hue="method",
    # complementary=True,
    # common_norm=False,
    # stat="count",
    stat="density",
    element="step",
    # fill=False,
    # bins=50,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0, 1),
    # ylim=(0, 1400),
    title="How much do the Unique Novel Valid linkers resemble the old linkers?",
    xlabel="Linker SuCOS",
)
plt.savefig("plots/conditional_frags_comparison_to_linker.png")

## Basic metrics

In [ ]:
metrics = {
    "ensemble_avg_energy": "Ensemble Average Energy",
    "mol_pred_energy": "Molecular Prediction Energy",
    "energy_ratio": "Energy Ratio",
    "sa": "Synthetic Accessability Score",
    "sa_normalized": "Synthetic Accessability Score (normalized)",
    "spacial": "Spacial Score",
    "qed": "Quantitative Estimation of Drug-likeness",
    "logp": "LogP",
    "lipinski": "Lipinski Rule of 5",
    "num_heavy": "Number of Heavy Atoms",
    "weight": "Molecular Weight",
    "num_rings": "Number of Rings",
}

In [ ]:
metric = "sa"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 10)
plt.savefig(f"plots/conditional_frags_{metric}.png")

In [ ]:
metric = "spacial"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 120)
plt.savefig(f"plots/conditional_frags_{metric}.png")

In [ ]:
metric = "qed"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 1)
plt.savefig(f"plots/conditional_frags_{metric}.png")

In [ ]:
metric = "energy_ratio"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    bins=100,
    hue="method",
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    log_scale=True if metric == "energy_ratio" else False,
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0.5, 200)
plt.savefig(f"plots/conditional_frags_{metric}.png")

In [ ]:
metric = "num_heavy"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    bins=np.array(range(int(df[df.valid]["num_heavy"].max()) + 1)),
    hue="method",
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    fill=True,
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 60)
plt.savefig(f"plots/conditional_frags_{metric}.png")

# Appendix

## Old plots

In [ ]:
sns.histplot(
    df, x="tanimoto_frag", hue="method", element="step", common_norm=False, bins=100
)

In [ ]:
sns.histplot(
    df, x="tanimoto_link", hue="method", element="step", common_norm=False, bins=100
)

In [ ]:
# metric = "sucos"
# name = metrics[metric]

# g = sns.FacetGrid(df, col="comparison", hue="method", height=5, aspect=1.3)
# g.map(
#     sns.histplot,
#     metric,
#     bins=50,
#     common_norm=False,
#     stat="density",
#     element="step",
#     # kde=True,
#     fill=False,
# )
# g.add_legend()

In [ ]:
# metric = "tanimoto"
# name = metrics[metric]

# g = sns.FacetGrid(df, col="comparison", hue="method", height=5, aspect=1.3)
# g.map(
#     sns.histplot,
#     metric,
#     bins=50,
#     common_norm=False,
#     stat="density",
#     element="step",
#     # kde=True,
#     fill=False,
# )
# g.add_legend()

In [ ]:
# g = sns.FacetGrid(df, col="method", row="comparison", height=3, aspect=1.3)
# g.map(sns.kdeplot, "num_atoms_cond", "num_atoms_pred")